# AIE464 Final: cross-domain benchmark (3 sources x 5 models)

Generated by `notebooks/build_cross_domain_colab.py`; edit the builder, not this file. Train on Kaggle / Augraphy / DIBCO, test zero-shot on all three test sets. Run cells in order. Full training starts only after the pilot is approved.

In [ ]:
# 1. Config: every setting lives here
DRIVE_ROOT = '/content/drive/MyDrive/AIE464_runs'   # same runs/ as the final notebook: Kaggle jobs are reused
REPO_URL = 'https://github.com/chayanon25/aie464-text-preserving-denoising.git'
GIT_SHA = 'main'          # set to the exact commit sha before running jobs
JOB_ID = None             # None = claim the next pending job from runs/jobs.json
OWNER = 'account-1'       # a label for this Colab account
SMOKE = False            # True = tiny synthetic CPU run, no Drive
MAX_MINUTES = 330         # pause safely before Colab kills the session (rerun to resume)
LOCAL_REPO = None

# Budgets (epochs, samples_per_epoch) mirror the Kaggle jobs; confirm after the pilot.
# They are set in cell 2b from src.experiments.CROSS_DOMAIN_BUDGETS (teacher before KD student).


In [ ]:
# 2. Mount Drive, clone the repo at GIT_SHA, put mini_project/ on sys.path
import os, subprocess, sys
from pathlib import Path
if SMOKE:
    MINI = Path(LOCAL_REPO)
    RUNS_ROOT = Path('runs').resolve()
else:
    from google.colab import drive
    drive.mount('/content/drive')
    if not Path('/content/repo').exists():
        subprocess.run(['git', 'clone', '-q', REPO_URL, '/content/repo'], check=True)
    subprocess.run(['git', '-C', '/content/repo', 'fetch', '-q', 'origin'], check=True)
    subprocess.run(['git', '-C', '/content/repo', 'checkout', '-q', GIT_SHA], check=True)
    MINI = Path('/content/repo/mini_project')
    RUNS_ROOT = Path(DRIVE_ROOT) / 'runs'
RUNS_ROOT.mkdir(parents=True, exist_ok=True)
sys.path.insert(0, str(MINI))
SHA = subprocess.run(['git', '-C', str(MINI), 'rev-parse', 'HEAD'], capture_output=True, text=True).stdout.strip() or None
print('code sha:', SHA, '| runs:', RUNS_ROOT)


In [ ]:
# 2b. Budgets and the cross-domain job list (needs the repo on sys.path)
from src.experiments import CROSS_DOMAIN_BUDGETS, CROSS_DOMAIN_MODELS, cross_domain_job_id
from src.domains import DOMAINS
BUDGETS = dict(CROSS_DOMAIN_BUDGETS)
SMOKE_JOBS = ['xd_augraphy_unet', 'xd_augraphy_kd']
MATRIX_JOBS = [cross_domain_job_id(s, m) for s in DOMAINS for m in CROSS_DOMAIN_MODELS]
print(len(BUDGETS), 'new jobs to train |', len(MATRIX_JOBS), 'models in the matrix')


In [ ]:
# 3. Extra packages (Colab already ships torch; never reinstall it)
if not SMOKE:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'augraphy', 'scikit-image'], check=True)


In [ ]:
# 4. Data: unzip the Drive bundle, build the Augraphy train/val set, run the fast tests
import torch
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
MANIFEST = MINI / 'outputs' / 'phase1' / 'split_manifest.json'
if SMOKE:
    from src.smoke import smoke_data
    SMOKE_SETS = smoke_data()   # no Augraphy generation in smoke mode
else:
    DATA_ROOT = Path('/content/data')
    if not (DATA_ROOT / 'raw').exists():
        subprocess.run(['unzip', '-q', '-o', f'{DRIVE_ROOT}/data/aie464_data.zip', '-d', str(DATA_ROOT)], check=True)
    os.environ['MIDTERM_CKPT_DIR'] = str(DATA_ROOT / 'checkpoints')
    os.environ['NAFNET_WEIGHTS'] = str(DATA_ROOT / 'nafnet' / 'NAFNet-SIDD-width32.pth')
    for sub in ('raw', 'external'):
        link = MINI / 'data' / sub
        link.parent.mkdir(exist_ok=True)
        if not link.exists():
            link.symlink_to(DATA_ROOT / sub)
    # Augraphy train/val from Kaggle train/val pages only (seed 7, no Folding); deterministic.
    from src.degrade import make_augraphy_split
    from src.domains import AUGRAPHY_TRAIN_DIR
    for split in ('train', 'val'):
        out = DATA_ROOT / AUGRAPHY_TRAIN_DIR / split
        if not (out / 'manifest.json').exists():
            info = make_augraphy_split(DATA_ROOT / 'raw', MANIFEST, out, split)
            print('augraphy', split, len(info['files']), 'pairs')
    _t = subprocess.run([sys.executable, '-m', 'pytest', '-q', '-x', '-m', 'not slow'], cwd=MINI, capture_output=True, text=True)
    print(_t.stdout[-4000:], _t.stderr[-2000:])
    assert _t.returncode == 0, 'fast test suite failed (see output above)'

def get_job_data(cfg):
    if SMOKE:
        return SMOKE_SETS[cfg.data]
    from src.domains import load_domain_train
    return load_domain_train(cfg.data, DATA_ROOT, MANIFEST)

def get_testsets():
    if SMOKE:
        return SMOKE_SETS['testsets']
    from src.eval_suite import load_testsets
    return load_testsets(DATA_ROOT / 'raw', MANIFEST,
                         DATA_ROOT / 'external' / 'augraphy_shift', DATA_ROOT / 'external' / 'dibco',
                         required=('kaggle_test', 'augraphy_shift', 'dibco2017'))
print('device:', DEVICE)


In [ ]:
# 5. Register budgeted jobs in the shared registry (idempotent)
from src.experiments import JOBS, with_budget
from src.jobs import JobRegistry
REGISTRY = JobRegistry(RUNS_ROOT / 'jobs.json')
SMOKE_OVERRIDES = dict(batch_size=2, patch=64, amp=False)

def budgeted(job_id):
    if SMOKE:
        return with_budget(job_id, 1, 4, git_sha=SHA, **SMOKE_OVERRIDES)
    epochs, samples = BUDGETS[job_id]
    return with_budget(job_id, epochs, samples, git_sha=SHA)

for job_id in (SMOKE_JOBS if SMOKE else BUDGETS):
    REGISTRY.add(budgeted(job_id))
print(REGISTRY.summary())


In [ ]:
# 6. Run jobs back to back: JOB_ID set = that job only, None = every pending job in BUDGETS order.
# Checkpoint + heartbeat every epoch; pauses before MAX_MINUTES. If Colab drops, rerun cells 1-6 to resume.
import time
from src.train_engine import run_job
started = time.time()
while True:
    if JOB_ID:
        cfg = budgeted(JOB_ID)
        REGISTRY.add(cfg)
        if REGISTRY.status(JOB_ID) == 'done':
            print(JOB_ID, 'already done'); break
        REGISTRY.mark(JOB_ID, 'running', owner=OWNER)
    else:
        cfg = REGISTRY.claim_next(OWNER)
    if cfg is None:
        print('No pending job: all done.'); break
    data, result = get_job_data(cfg), None
    while (left := MAX_MINUTES - (time.time() - started) / 60) > 0:
        result = run_job(cfg, RUNS_ROOT / cfg.job_id, data, DEVICE, max_minutes=left, max_epochs=1,
                         runs_root=RUNS_ROOT, on_progress=lambda: REGISTRY.heartbeat(cfg.job_id, owner=OWNER))
        REGISTRY.heartbeat(cfg.job_id, owner=OWNER)
        last = result['history'][-1] if result['history'] else {}
        print(cfg.job_id, result['status'], last)
        if result['status'] == 'done':
            REGISTRY.mark(cfg.job_id, 'done', owner=OWNER)
            break
    if JOB_ID or not (result and result['status'] == 'done'):
        print('stopped at', cfg.job_id, 'status:', result and result['status'],
              '(time budget reached: rerun cells 1-6 to continue)' if left <= 0 else ''); break
print(REGISTRY.summary())


In [ ]:
# 7. Evaluate all 15 matrix models on all 3 test sets into eval_xd/ (fresh JSON with FM; eval/ untouched)
import json
from src.eval_suite import evaluate, predict_fn, save_eval
from src.train_engine import load_trained
TESTSETS = get_testsets()
EVAL_TILE = 1024
models, missing = {}, []
for job_id in MATRIX_JOBS:
    (models.__setitem__(job_id, job_id) if (RUNS_ROOT / job_id / 'best.pt').exists() else missing.append(job_id))
for name, ref in models.items():
    out = RUNS_ROOT / 'eval_xd' / f'{name}.json'
    if out.exists():
        continue
    model = load_trained(ref, RUNS_ROOT, DEVICE)
    save_eval(evaluate(predict_fn(model, DEVICE, tile=EVAL_TILE, amp=False), TESTSETS, name), out)
    print('evaluated', name)
print('not trained yet:', missing)


In [ ]:
# 8. Latency: warm-up 10, measure 50, median ms + peak VRAM + params (540x420 page)
from src.infer import measure_latency
latency_path = RUNS_ROOT / 'latency.json'
latency = json.loads(latency_path.read_text()) if latency_path.exists() else {}
for name, ref in models.items():
    if name not in latency:
        model = load_trained(ref, RUNS_ROOT, DEVICE)
        latency[name] = measure_latency(model, (1, 1, 540, 420), DEVICE,
                                        warmup=2 if SMOKE else 10, iters=3 if SMOKE else 50)
        latency[name]['device'] = torch.cuda.get_device_name() if DEVICE == 'cuda' else 'cpu'
latency_path.write_text(json.dumps(latency, indent=2))
print(json.dumps(latency, indent=2))


In [ ]:
# 9. Cross-domain tables (PSNR, FM, transfer gap) -> runs/cross_domain.md
from src.cross_domain import write_cross_domain_report
write_cross_domain_report(RUNS_ROOT / 'eval_xd', RUNS_ROOT / 'cross_domain.md')
print((RUNS_ROOT / 'cross_domain.md').read_text())
